In [1]:
import os

In [2]:
%pwd

'/Users/venkatsaiganeshtadiboina/nlp_text_summarization_app/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/venkatsaiganeshtadiboina/nlp_text_summarization_app'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [11]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)

        self.params = read_yaml(params_filepath)

        create_directories(
            [self.config.artifacts_root]
        )

    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path = config.model_path,
            tokenizer_path= config.tokenizer_path,
            metric_file_name=config.metric_file_name
        )

        return model_evaluation_config

In [12]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from datasets import (
    load_dataset,
    load_from_disk
)

from evaluate import load

import torch
import pandas as pd

from tqdm import tqdm

In [13]:
import torch
import pandas as pd

from tqdm import tqdm
from evaluate import load

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer
)

from datasets import load_from_disk


class ModelEvaluation:

    def __init__(self, config: ModelEvaluationConfig):
        self.config = config


    def generate_batch_sized_chunks(
        self,
        list_of_elements,
        batch_size
    ):
        """
        split the dataset into smaller batches that we can process simultaneously
        Yield successive batch-sized chunks from list_of_elements.
        """

        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i: i + batch_size]


    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=2,
        device="cpu",
        column_text="dialogue",
        column_summary="summary"
    ):

        article_batches = list(
            self.generate_batch_sized_chunks(
                dataset[column_text],
                batch_size
            )
        )

        target_batches = list(
            self.generate_batch_sized_chunks(
                dataset[column_summary],
                batch_size
            )
        )


        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=len(article_batches)
        ):

            inputs = tokenizer(
                article_batch,
                max_length=1024,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )


            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                length_penalty=0.8,
                num_beams=8,
                max_length=128
            )


            decoded_summaries = [
                tokenizer.decode(
                    s,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True
                )
                for s in summaries
            ]


            decoded_summaries = [
                d.replace("", " ")
                for d in decoded_summaries
            ]


            metric.add_batch(
                predictions=decoded_summaries,
                references=target_batch
            )


        score = metric.compute()

        return score


    def evaluate(self):

        # FORCE CPU FOR MAC STABILITY
        device = "cpu"


        tokenizer = AutoTokenizer.from_pretrained(
            self.config.tokenizer_path
        )


        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_path
        ).to(device)


        # loading data
        dataset_samsum_pt = load_from_disk(
            self.config.data_path
        )


        rouge_metric = load("rouge")


        score = self.calculate_metric_on_test_ds(
            dataset=dataset_samsum_pt["test"][0:10],
            metric=rouge_metric,
            model=model_pegasus,
            tokenizer=tokenizer,
            batch_size=2,
            column_text="dialogue",
            column_summary="summary"
        )


        rouge_dict = {
            rn: score[rn]
            for rn in score.keys()
        }


        df = pd.DataFrame(
            [rouge_dict],
            index=["pegasus"]
        )


        df.to_csv(
            self.config.metric_file_name,
            index=False
        )


        print("\nModel evaluation completed successfully.")

In [14]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluate()
except Exception as e:
    raise e

[2026-05-25 11:23:29,122: INFO: common: yaml file: /Users/venkatsaiganeshtadiboina/nlp_text_summarization_app/config/config.yaml loaded successfully]
[2026-05-25 11:23:29,125: INFO: common: yaml file: /Users/venkatsaiganeshtadiboina/nlp_text_summarization_app/params.yaml loaded successfully]
[2026-05-25 11:23:29,125: INFO: common: created directory at: artifacts]
[2026-05-25 11:23:29,126: INFO: common: created directory at: artifacts/model_evaluation]


100%|██████████| 5/5 [00:33<00:00,  6.64s/it]

[2026-05-25 11:24:05,527: INFO: rouge_scorer: Using default tokenizer.]

Model evaluation completed successfully.
